In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from show import *
from utils import *


In [2]:
def remap(data, header, nx=720, ny=1800, **kwargs):
    from transforms import ToSpherical
    from scipy.ndimage import map_coordinates

    view = View.from_header(header).update(**kwargs)
    transform = ~ToSpherical() - view.to_carrington(**kwargs)

    lat_spread = 2
    dlon = 360 / ny
    dlat = lat_spread / (nx - 1)


    grid = np.mgrid[-lat_spread / 2:lat_spread / 2 + dlat / 2:dlat, :360:dlon]
    grid[0] = np.arcsin(grid[0].clip(-1,1)) * 180 / np.pi

    grid, alpha = transform(grid)
    data_ = map_coordinates(np.nan_to_num(data), grid, cval=np.nan)

    return data_, grid, alpha


In [3]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])

#t = np.where(np.all([dates > datetime(2026,4,9),
#                     dates < datetime(2026,5,1)], axis=0))[0] # CR2310

#t = np.where(np.all([dates > datetime(2026,6,24),
#                     dates < datetime(2026,7,22)], axis=0))[0] # CR2313

#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,8,31)], axis=0))[0] # CR2314

t = np.where(np.all([dates > datetime(2026,8,31),
                     dates < datetime(2026,9,29)], axis=0))[0] # CR2315

#t = np.where(np.all([dates > datetime(2026,7,8),
#                     dates < datetime(2026,8,8)], axis=0))[0]

#t = np.where(np.all([dates > datetime(2026,9,14),
#                     dates < datetime(2026,10,13)], axis=0))[0]



files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20260831T113003_V202610191358C_0648310501.fits'),
 np.str_('/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20260928T093003_V202610191523C_0649280504.fits'),
 83)

In [6]:
N = 0

D = 0
B = 0
C = 0

A = 0
U = 0

for file in files[:]:

    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    x0, y0 = header['PXBEG2'] - 1, header['PXBEG1'] - 1
    f, grid, alpha = remap(data, header, delta_t=20, correct_dr=True, minmu=0.1)

    x, y = grid
    x, y = (x + x0) / 1024 - 1, (y + y0) / 1024 - 1
    z = 1 / alpha

    d = np.moveaxis(np.array([[x, y, np.ones_like(x)]]), (0,1), (-2,-1))
    b = d * np.moveaxis(np.array([[z]]), (0,1), (-2,-1))
    c = d * np.moveaxis(np.array([[f]]), (0,1), (-2,-1))
    d = np.swapaxes(d, -2, -1) @ d

    a = np.moveaxis(np.array([[z]]), (0,1), (-2,-1))
    u = a * np.moveaxis(np.array([[f]]), (0,1), (-2,-1))
    a = a * a

    n = (~np.isnan(f)).astype(float)
    N += np.nan_to_num(n)

    D += np.nan_to_num((d - D) * np.expand_dims(n / N, (-2, -1)))
    B += np.nan_to_num((b - B) * np.expand_dims(n / N, (-2, -1)))
    C += np.nan_to_num((c - C) * np.expand_dims(n / N, (-2, -1)))

    A += np.nan_to_num((a - A) * np.expand_dims(n / N, (-2, -1)))
    U += np.nan_to_num((u - U) * np.expand_dims(n / N, (-2, -1)))



A[N == 0] = np.nan
U[N == 0] = np.nan

D[N == 0] = np.nan
B[N == 0] = np.nan
C[N == 0] = np.nan
N[N == 0] = np.nan

/tmp/ipykernel_132070/1859718329.py:35: RuntimeWarning: invalid value encountered in divide
  D += np.nan_to_num((d - D) * np.expand_dims(n / N, (-2, -1)))
/tmp/ipykernel_132070/1859718329.py:36: RuntimeWarning: invalid value encountered in divide
  B += np.nan_to_num((b - B) * np.expand_dims(n / N, (-2, -1)))
/tmp/ipykernel_132070/1859718329.py:37: RuntimeWarning: invalid value encountered in divide
  C += np.nan_to_num((c - C) * np.expand_dims(n / N, (-2, -1)))
/tmp/ipykernel_132070/1859718329.py:39: RuntimeWarning: invalid value encountered in divide
  A += np.nan_to_num((a - A) * np.expand_dims(n / N, (-2, -1)))
/tmp/ipykernel_132070/1859718329.py:40: RuntimeWarning: invalid value encountered in divide
  U += np.nan_to_num((u - U) * np.expand_dims(n / N, (-2, -1)))


In [7]:
A_ = 1 / A
Q = A_ * U

#W = 1
#W = 1 / np.expand_dims(Z2, (-2,-1))
W = np.ones_like(A)
#W[360:] = np.nan


BT = np.swapaxes(B, -2, -1)


H = np.nanmean((D - A_ * BT @ B) * W, axis=(0,1))
H = np.linalg.inv(H)


CT = np.swapaxes(C, -2, -1)
V = np.nanmean((CT - BT * Q) * W, axis=(0,1))

P = H @ V
Q_ = Q - A_ * B @ P

P = np.squeeze(P)
Q = np.squeeze(Q)
Q_ = np.squeeze(Q_)

print(P)

[-1.02014401 -0.05979934 -0.21628388]


In [8]:
show_map(Q_, vmin=-500, vmax=500, sine_lat=True)

In [9]:
nx = len(Q)

lat = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

#flux = np.nanmean(Q * Z2, axis=1) / np.nanmean(Z2, axis=1)
#flux_ = np.nanmean(Q_ * Z2, axis=1) / np.nanmean(Z2, axis=1)

flux = np.nanmean(Q, axis=1)
flux_ = np.nanmean(Q_, axis=1)

plt.figure(figsize=(10,8))
plt.plot(lat, flux)
plt.plot(lat, flux_)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

/tmp/ipykernel_132070/1439491012.py:8: RuntimeWarning: Mean of empty slice
  flux = np.nanmean(Q, axis=1)
/tmp/ipykernel_132070/1439491012.py:9: RuntimeWarning: Mean of empty slice
  flux_ = np.nanmean(Q_, axis=1)


In [321]:
file = '/home/ulyanov/data/sdo/hmi/synoptic_maps/hmi.Synoptic_Mr_720s.2310.synopMr.fits'

with fits.open(file) as hdul:
    header = hdul[0].header.copy()
    data_ = hdul[0].data.copy()

data_ = rebin(data_, 2)
#data_ = gaussian_filter(data_, 2, truncate=3)
header['T_OBS']

'2026.04.29_07:08:43_TAI'

In [322]:
show_map(data_, vmin=-500, vmax=500, sine_lat=True)

In [323]:
#flux_hmi = np.nanmean(data_ * Z2, axis=1) / np.nanmean(Z2, axis=1)
flux_hmi = np.nanmean(data_, axis=1)

nx, ny = data_.shape
lat_hmi = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

plt.figure(figsize=(10,8))
plt.plot(lat_hmi, flux_hmi, label='HMI')
plt.plot(lat, flux, label='FDT')
plt.plot(lat, flux_, label='FDT (corrected)', color='red')

plt.xlabel('Latitude, degrees')
plt.ylabel(r'Flux, Mx/cm$^2$')
plt.xlim(-90,90)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.legend()
plt.tight_layout()

/tmp/ipykernel_122126/4225042231.py:2: RuntimeWarning: Mean of empty slice
  flux_hmi = np.nanmean(data_, axis=1)
